# CleanAir AI — Notebook 02: Weather and Geospatial Integration

This notebook adds weather and location intelligence to the live AQI station data produced by Notebook 01.

## Inputs

- `data/processed/station_intelligence_latest.csv`

## Main tasks

1. Load latest station-level AQI data.
2. Fetch real weather data from Open-Meteo.
3. Add temperature, humidity, wind speed, rainfall, pressure, and cloud cover.
4. Calculate a weather trapping score.
5. Add geospatial proxy features.
6. Estimate land-use and urban pressure around stations.
7. Save the enriched station dataset.

## Main output

- `data/processed/station_weather_geospatial.csv`

In [2]:
from pathlib import Path
from datetime import datetime
import time
import json
import warnings

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROJECT_ROOT / "data" / "cache"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = PROJECT_ROOT / "reports"

for folder in [
    PROCESSED_DIR,
    CACHE_DIR,
    OUTPUTS_DIR,
    REPORTS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed folder:", PROCESSED_DIR)
print("Cache folder:", CACHE_DIR)

Project root: C:\Users\Lenovo\Desktop\CleanAir_AI
Processed folder: C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed
Cache folder: C:\Users\Lenovo\Desktop\CleanAir_AI\data\cache


In [4]:
STATION_INPUT_PATH = PROCESSED_DIR / "station_intelligence_latest.csv"

if not STATION_INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Missing input file: {STATION_INPUT_PATH}. "
        "Run Notebook 01 first."
    )

station_df = pd.read_csv(STATION_INPUT_PATH)

print("Loaded station data.")
print("Shape:", station_df.shape)

display(station_df.head())

Loaded station data.
Shape: (500, 32)


,station_name,city,state,lat,lon,timestamp,PM2.5,PM10,NO2,SO2,...,aqi_category,snapshot_fetch_time,station_id,valid_pollutant_count,has_particulate,aqi_is_valid,aqi_quality_flag,reported_aqi,reported_aqi_category,reported_dominant_pollutant
0,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-16 05:00:00,54.0,172.0,32.0,16.0,...,Severe,2026-07-16 05:38:19,andhra_pradesh__amaravati__secretariat_amarava...,7,True,True,Valid AQI,500.00,Severe,CO
1,"Gulzarpet, Anantapur - APPCB",Anantapur,Andhra Pradesh,14.675886,77.593027,2026-07-16 05:00:00,85.0,76.0,16.0,11.0,...,Very Poor,2026-07-16 05:38:19,andhra_pradesh__anantapur__gulzarpet_anantapur...,7,True,True,Valid AQI,329.70,Very Poor,CO
2,"Gangineni Cheruvu, Chittoor - APPCB",Chittoor,Andhra Pradesh,13.204880,79.097889,2026-07-16 05:00:00,70.0,64.0,31.0,21.0,...,Very Poor,2026-07-16 05:38:19,andhra_pradesh__chittoor__gangineni_cheruvu_ch...,7,True,True,Valid AQI,323.85,Very Poor,CO
3,"District Court, Eluru - APPCB",Eluru,Andhra Pradesh,16.711754,81.092095,2026-07-16 05:00:00,49.0,49.0,22.0,13.0,...,Very Poor,2026-07-16 05:38:19,andhra_pradesh__eluru__district_court_eluru_appcb,7,True,True,Valid AQI,306.27,Very Poor,CO
4,"Rajendra Nagar North, Guntur - APPCB",Guntur,Andhra Pradesh,16.316553,80.413302,2026-07-16 05:00:00,52.0,52.0,42.0,14.0,...,Poor,2026-07-16 05:38:19,andhra_pradesh__guntur__rajendra_nagar_north_g...,7,True,True,Valid AQI,271.30,Poor,CO


In [5]:
required_columns = [
    "station_id",
    "station_name",
    "city",
    "state",
    "lat",
    "lon",
    "timestamp",
    "reported_aqi",
    "reported_aqi_category",
    "reported_dominant_pollutant"
]

missing_columns = [
    col for col in required_columns
    if col not in station_df.columns
]

if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

station_df["lat"] = pd.to_numeric(station_df["lat"], errors="coerce")
station_df["lon"] = pd.to_numeric(station_df["lon"], errors="coerce")

station_df = station_df.dropna(subset=["lat", "lon"]).copy()

print("Required columns available.")
print("Usable stations with coordinates:", len(station_df))

Required columns available.
Usable stations with coordinates: 500


In [6]:
print("States:", station_df["state"].nunique())
print("Cities:", station_df["city"].nunique())
print("Stations:", station_df["station_id"].nunique())

print("\nTop states by station count:")
print(station_df["state"].value_counts().head(15))

print("\nAQI category distribution:")
print(station_df["reported_aqi_category"].value_counts(dropna=False))

States: 29
Cities: 264
Stations: 500

Top states by station count:
state
Maharashtra       82
Uttar Pradesh     60
Rajasthan         46
Delhi             44
Bihar             33
Haryana           30
Karnataka         27
Madhya Pradesh    25
West Bengal       22
Gujarat           20
Odisha            18
Andhra Pradesh    17
Chhattisgarh      14
Tamil Nadu        12
Telangana         12
Name: count, dtype: int64

AQI category distribution:
reported_aqi_category
Very Poor       183
Severe          147
Poor             84
Moderate         49
Unknown          22
Satisfactory     12
Good              3
Name: count, dtype: int64


In [7]:
OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"

def fetch_weather_for_station(lat, lon):
    """
    Fetch current weather from Open-Meteo for one station.
    No API key required.
    """

    params = {
        "latitude": lat,
        "longitude": lon,
        "current": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "rain",
            "wind_speed_10m",
            "wind_direction_10m",
            "surface_pressure",
            "cloud_cover"
        ],
        "timezone": "auto"
    }

    response = requests.get(
        OPEN_METEO_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    payload = response.json()

    current = payload.get("current", {})

    return {
        "weather_time": current.get("time"),
        "temperature_2m": current.get("temperature_2m"),
        "relative_humidity_2m": current.get("relative_humidity_2m"),
        "precipitation": current.get("precipitation"),
        "rain": current.get("rain"),
        "wind_speed_10m": current.get("wind_speed_10m"),
        "wind_direction_10m": current.get("wind_direction_10m"),
        "surface_pressure": current.get("surface_pressure"),
        "cloud_cover": current.get("cloud_cover")
    }

print("Weather function ready.")

Weather function ready.


In [8]:
sample_station = station_df.iloc[0]

sample_weather = fetch_weather_for_station(
    lat=sample_station["lat"],
    lon=sample_station["lon"]
)

print("Sample station:")
print(sample_station["station_name"], "-", sample_station["city"], sample_station["state"])

print("\nWeather response:")
print(json.dumps(sample_weather, indent=4))

Sample station:
Secretariat, Amaravati - APPCB - Amaravati Andhra Pradesh

Weather response:
{
    "weather_time": "2026-07-16T10:30",
    "temperature_2m": 31.9,
    "relative_humidity_2m": 55,
    "precipitation": 0.0,
    "rain": 0.0,
    "wind_speed_10m": 17.7,
    "wind_direction_10m": 298,
    "surface_pressure": 1004.7,
    "cloud_cover": 100
}


In [9]:
WEATHER_CACHE_PATH = CACHE_DIR / "open_meteo_weather_latest.csv"

weather_rows = []

for _, row in tqdm(
    station_df.iterrows(),
    total=len(station_df),
    desc="Fetching station weather"
):
    try:
        weather = fetch_weather_for_station(
            lat=row["lat"],
            lon=row["lon"]
        )

        weather["station_id"] = row["station_id"]
        weather["lat"] = row["lat"]
        weather["lon"] = row["lon"]

        weather_rows.append(weather)

        time.sleep(0.15)

    except Exception as exc:
        weather_rows.append({
            "station_id": row["station_id"],
            "lat": row["lat"],
            "lon": row["lon"],
            "weather_time": np.nan,
            "temperature_2m": np.nan,
            "relative_humidity_2m": np.nan,
            "precipitation": np.nan,
            "rain": np.nan,
            "wind_speed_10m": np.nan,
            "wind_direction_10m": np.nan,
            "surface_pressure": np.nan,
            "cloud_cover": np.nan,
            "weather_error": str(exc)
        })

weather_df = pd.DataFrame(weather_rows)

weather_df.to_csv(WEATHER_CACHE_PATH, index=False)

print("Weather data saved to:")
print(WEATHER_CACHE_PATH)

print("Weather shape:", weather_df.shape)
display(weather_df.head())

Fetching station weather:   0%|          | 0/500 [00:00<?, ?it/s]

Weather data saved to:
C:\Users\Lenovo\Desktop\CleanAir_AI\data\cache\open_meteo_weather_latest.csv
Weather shape: (500, 12)


,weather_time,temperature_2m,relative_humidity_2m,precipitation,rain,wind_speed_10m,wind_direction_10m,surface_pressure,cloud_cover,station_id,lat,lon
0,2026-07-16T10:30,31.9,55,0.0,0.0,17.7,298,1004.7,100,andhra_pradesh__amaravati__secretariat_amarava...,16.515083,80.518167
1,2026-07-16T10:30,31.9,45,0.0,0.0,17.4,280,970.8,99,andhra_pradesh__anantapur__gulzarpet_anantapur...,14.675886,77.593027
2,2026-07-16T10:30,32.6,39,0.0,0.0,12.3,284,974.2,100,andhra_pradesh__chittoor__gangineni_cheruvu_ch...,13.204880,79.097889
3,2026-07-16T10:30,31.2,62,0.0,0.0,13.6,294,1004.6,100,andhra_pradesh__eluru__district_court_eluru_appcb,16.711754,81.092095
4,2026-07-16T10:30,31.9,54,0.0,0.0,16.5,295,1001.9,100,andhra_pradesh__guntur__rajendra_nagar_north_g...,16.316553,80.413302


In [10]:
print("Missing weather values:")

weather_columns = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "wind_speed_10m",
    "wind_direction_10m",
    "surface_pressure",
    "cloud_cover"
]

print(weather_df[weather_columns].isna().sum())

print("\nWeather rows:", len(weather_df))

Missing weather values:
temperature_2m          0
relative_humidity_2m    0
precipitation           0
rain                    0
wind_speed_10m          0
wind_direction_10m      0
surface_pressure        0
cloud_cover             0
dtype: int64

Weather rows: 500


In [11]:
missing_weather_ids = weather_df[
    weather_df[
        [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "rain",
            "wind_speed_10m",
            "wind_direction_10m",
            "surface_pressure",
            "cloud_cover"
        ]
    ].isna().any(axis=1)
]["station_id"].tolist()

print("Stations with missing weather:", len(missing_weather_ids))

display(
    station_df[
        station_df["station_id"].isin(missing_weather_ids)
    ][
        [
            "state",
            "city",
            "station_name",
            "lat",
            "lon",
            "reported_aqi",
            "reported_aqi_category"
        ]
    ]
)

Stations with missing weather: 0


,state,city,station_name,lat,lon,reported_aqi,reported_aqi_category


In [12]:
station_weather_df = station_df.merge(
    weather_df.drop(columns=["lat", "lon"], errors="ignore"),
    on="station_id",
    how="left"
)

print("Merged station + weather shape:", station_weather_df.shape)
display(station_weather_df.head())

Merged station + weather shape: (500, 41)


,station_name,city,state,lat,lon,timestamp,PM2.5,PM10,NO2,SO2,...,reported_dominant_pollutant,weather_time,temperature_2m,relative_humidity_2m,precipitation,rain,wind_speed_10m,wind_direction_10m,surface_pressure,cloud_cover
0,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-16 05:00:00,54.0,172.0,32.0,16.0,...,CO,2026-07-16T10:30,31.9,55,0.0,0.0,17.7,298,1004.7,100
1,"Gulzarpet, Anantapur - APPCB",Anantapur,Andhra Pradesh,14.675886,77.593027,2026-07-16 05:00:00,85.0,76.0,16.0,11.0,...,CO,2026-07-16T10:30,31.9,45,0.0,0.0,17.4,280,970.8,99
2,"Gangineni Cheruvu, Chittoor - APPCB",Chittoor,Andhra Pradesh,13.204880,79.097889,2026-07-16 05:00:00,70.0,64.0,31.0,21.0,...,CO,2026-07-16T10:30,32.6,39,0.0,0.0,12.3,284,974.2,100
3,"District Court, Eluru - APPCB",Eluru,Andhra Pradesh,16.711754,81.092095,2026-07-16 05:00:00,49.0,49.0,22.0,13.0,...,CO,2026-07-16T10:30,31.2,62,0.0,0.0,13.6,294,1004.6,100
4,"Rajendra Nagar North, Guntur - APPCB",Guntur,Andhra Pradesh,16.316553,80.413302,2026-07-16 05:00:00,52.0,52.0,42.0,14.0,...,CO,2026-07-16T10:30,31.9,54,0.0,0.0,16.5,295,1001.9,100


In [13]:
numeric_weather_columns = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "wind_speed_10m",
    "wind_direction_10m",
    "surface_pressure",
    "cloud_cover"
]

for col in numeric_weather_columns:
    station_weather_df[col] = pd.to_numeric(
        station_weather_df[col],
        errors="coerce"
    )

print("Weather columns converted to numeric.")

Weather columns converted to numeric.


In [14]:
for col in numeric_weather_columns:
    station_weather_df[col] = station_weather_df.groupby("state")[col].transform(
        lambda x: x.fillna(x.median())
    )

    station_weather_df[col] = station_weather_df[col].fillna(
        station_weather_df[col].median()
    )

print("Missing weather values after imputation:")

print(
    station_weather_df[numeric_weather_columns]
    .isna()
    .sum()
)

Missing weather values after imputation:
temperature_2m          0
relative_humidity_2m    0
precipitation           0
rain                    0
wind_speed_10m          0
wind_direction_10m      0
surface_pressure        0
cloud_cover             0
dtype: int64


In [15]:
def normalize_series(series):
    """
    Normalize a numeric pandas Series to 0-1.
    Handles constant and empty values safely.
    """

    series = pd.to_numeric(series, errors="coerce")

    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or min_val == max_val:
        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return (series - min_val) / (max_val - min_val)


def calculate_weather_trapping_score(df):
    """
    Higher score means weather conditions may trap pollution.

    Logic:
    - Low wind speed increases trapping.
    - High humidity increases particle persistence.
    - Low rainfall increases trapping.
    - High cloud cover can indicate stagnant conditions.
    """

    wind_norm = normalize_series(df["wind_speed_10m"])
    humidity_norm = normalize_series(df["relative_humidity_2m"])
    rain_norm = normalize_series(df["rain"].fillna(0))
    cloud_norm = normalize_series(df["cloud_cover"])

    low_wind_score = 1 - wind_norm
    high_humidity_score = humidity_norm
    low_rain_score = 1 - rain_norm
    cloud_score = cloud_norm

    trapping_score = (
        0.40 * low_wind_score
        + 0.25 * high_humidity_score
        + 0.25 * low_rain_score
        + 0.10 * cloud_score
    )

    return (trapping_score * 100).round(2)

print("Weather trapping score function ready.")

Weather trapping score function ready.


In [16]:
station_weather_df["weather_trapping_score"] = calculate_weather_trapping_score(
    station_weather_df
)

def trapping_category(score):
    if pd.isna(score):
        return "Unknown"
    elif score < 30:
        return "Low trapping"
    elif score < 60:
        return "Moderate trapping"
    elif score < 80:
        return "High trapping"
    else:
        return "Very high trapping"


station_weather_df["weather_trapping_category"] = station_weather_df[
    "weather_trapping_score"
].apply(trapping_category)

print("Weather trapping score calculated.")

display(
    station_weather_df[
        [
            "state",
            "city",
            "station_name",
            "reported_aqi",
            "reported_aqi_category",
            "temperature_2m",
            "relative_humidity_2m",
            "rain",
            "wind_speed_10m",
            "cloud_cover",
            "weather_trapping_score",
            "weather_trapping_category"
        ]
    ].head(20)
)

Weather trapping score calculated.

,state,city,station_name,reported_aqi,reported_aqi_category,temperature_2m,relative_humidity_2m,rain,wind_speed_10m,cloud_cover,weather_trapping_score,weather_trapping_category
0,Andhra Pradesh,Amaravati,"Secretariat, Amaravati - APPCB",500.00,Severe,31.9,55,0.0,17.7,100,55.79,Moderate trapping
1,Andhra Pradesh,Anantapur,"Gulzarpet, Anantapur - APPCB",329.70,Very Poor,31.9,45,0.0,17.4,99,52.55,Moderate trapping
2,Andhra Pradesh,Chittoor,"Gangineni Cheruvu, Chittoor - APPCB",323.85,Very Poor,32.6,39,0.0,12.3,100,59.68,Moderate trapping
3,Andhra Pradesh,Eluru,"District Court, Eluru - APPCB",306.27,Very Poor,31.2,62,0.0,13.6,100,65.78,High trapping
4,Andhra Pradesh,Guntur,"Rajendra Nagar North, Guntur - APPCB",271.30,Poor,31.9,54,0.0,16.5,100,57.59,Moderate trapping
5,Andhra Pradesh,Kadapa,"Yerramukkapalli, Kadapa - APPCB",364.85,Very Poor,33.0,44,0.0,17.9,99,51.28,Moderate trapping
6,Andhra Pradesh,Kurnool,"Sita Rama Nagar, Kurnool - APPCB",174.94,Moderate,33.0,46,0.0,18.5,38,44.83,Moderate trapping
7,Andhra Pradesh,Machilipatnam,"Srinivas Nagar Colony, Machilipatnam - APPCB",317.99,Very Poor,33.6,48,0.0,20.8,100,47.60,Moderate trapping
8,Andhra Pradesh,Nellore,"Ambedkar Nagar, Nellore - APPCB",412.83,Severe,34.7,42,0.0,18.3,100,49.92,Moderate trapping
9,Andhra Pradesh,Rajamahendravaram,"Anand Kala Kshetram, Rajamahendravaram - APPCB",353.14,Very Poor,30.8,64,0.0,17.3,100,59.82,Moderate trapping


In [17]:
def classify_india_region(lat, lon):
    """
    Rough geographic region classification for India.
    This is a coarse proxy, not administrative truth.
    """

    if pd.isna(lat) or pd.isna(lon):
        return "Unknown"

    if lat >= 28:
        return "North India"
    elif lat <= 15:
        return "South India"
    elif lon >= 85:
        return "East India"
    elif lon <= 73:
        return "West India"
    else:
        return "Central India"


station_weather_df["geo_region_proxy"] = station_weather_df.apply(
    lambda row: classify_india_region(row["lat"], row["lon"]),
    axis=1
)

print("Region proxy added.")
print(station_weather_df["geo_region_proxy"].value_counts())

Region proxy added.
geo_region_proxy
Central India    208
North India      128
East India        75
West India        49
South India       40
Name: count, dtype: int64


In [18]:
metro_cities = {
    "Delhi",
    "Mumbai",
    "Kolkata",
    "Chennai",
    "Bengaluru",
    "Bangalore",
    "Hyderabad",
    "Ahmedabad",
    "Pune",
    "Surat",
    "Jaipur",
    "Lucknow",
    "Kanpur",
    "Nagpur",
    "Indore",
    "Patna",
    "Bhopal",
    "Ludhiana",
    "Agra",
    "Nashik",
    "Vadodara",
    "Faridabad",
    "Ghaziabad",
    "Noida",
    "Gurugram",
    "Gurgaon"
}

industrial_keywords = [
    "industrial",
    "estate",
    "phase",
    "sector",
    "area",
    "sidcul",
    "midc",
    "gida",
    "baddi",
    "mandi gobindgarh",
    "dharuhera"
]

traffic_keywords = [
    "road",
    "crossing",
    "junction",
    "bus",
    "railway",
    "station",
    "traffic",
    "market",
    "circle",
    "chowk"
]


def infer_landuse_proxy(row):
    """
    Infer land-use from station and city names.
    This is a proxy and must be labeled as proxy in reports.
    """

    city = str(row.get("city", "")).strip()
    station = str(row.get("station_name", "")).strip().lower()
    combined_text = f"{city.lower()} {station}"

    if any(keyword in combined_text for keyword in industrial_keywords):
        return "Industrial / mixed urban proxy"

    if any(keyword in combined_text for keyword in traffic_keywords):
        return "Traffic / commercial proxy"

    if city in metro_cities:
        return "Dense urban proxy"

    return "General urban / semi-urban proxy"


station_weather_df["landuse_type_proxy"] = station_weather_df.apply(
    infer_landuse_proxy,
    axis=1
)

station_weather_df["landuse_data_source"] = "Proxy from station/city naming"

print("Land-use proxy added.")
print(station_weather_df["landuse_type_proxy"].value_counts())

Land-use proxy added.
landuse_type_proxy
General urban / semi-urban proxy    285
Dense urban proxy                   152
Industrial / mixed urban proxy       49
Traffic / commercial proxy           14
Name: count, dtype: int64


In [19]:
def calculate_urban_pressure_score(row):
    score = 0

    city = str(row.get("city", "")).strip()
    landuse = str(row.get("landuse_type_proxy", "")).lower()

    if city in metro_cities:
        score += 35

    if "traffic" in landuse:
        score += 30

    if "industrial" in landuse:
        score += 35

    if "dense urban" in landuse:
        score += 25

    if "general urban" in landuse:
        score += 15

    return min(score, 100)


station_weather_df["urban_pressure_score"] = station_weather_df.apply(
    calculate_urban_pressure_score,
    axis=1
)

def urban_pressure_category(score):
    if score < 25:
        return "Low urban pressure"
    elif score < 50:
        return "Moderate urban pressure"
    elif score < 75:
        return "High urban pressure"
    else:
        return "Very high urban pressure"


station_weather_df["urban_pressure_category"] = station_weather_df[
    "urban_pressure_score"
].apply(urban_pressure_category)

print("Urban pressure score calculated.")
print(station_weather_df["urban_pressure_category"].value_counts())

Urban pressure score calculated.
urban_pressure_category
Low urban pressure         285
High urban pressure        176
Moderate urban pressure     39
Name: count, dtype: int64


In [20]:
def calculate_road_density_proxy(row):
    """
    Proxy road-density score based on city size and station naming.
    Later this can be replaced with real OpenStreetMap road length.
    """

    score = 20

    city = str(row.get("city", "")).strip()
    station = str(row.get("station_name", "")).lower()
    landuse = str(row.get("landuse_type_proxy", "")).lower()

    if city in metro_cities:
        score += 35

    if "traffic" in landuse:
        score += 30

    if any(word in station for word in ["road", "junction", "circle", "chowk", "bus", "railway"]):
        score += 25

    if "industrial" in landuse:
        score += 10

    return min(score, 100)


station_weather_df["road_density_proxy"] = station_weather_df.apply(
    calculate_road_density_proxy,
    axis=1
)

def road_density_category(score):
    if score < 30:
        return "Low road density proxy"
    elif score < 60:
        return "Moderate road density proxy"
    elif score < 80:
        return "High road density proxy"
    else:
        return "Very high road density proxy"


station_weather_df["road_density_category"] = station_weather_df[
    "road_density_proxy"
].apply(road_density_category)

station_weather_df["road_density_data_source"] = "Proxy from city/station naming"

print("Road-density proxy calculated.")
print(station_weather_df["road_density_category"].value_counts())

Road-density proxy calculated.
road_density_category
Low road density proxy          285
Moderate road density proxy     183
High road density proxy          26
Very high road density proxy      6
Name: count, dtype: int64


In [21]:
aqi_norm = normalize_series(station_weather_df["reported_aqi"])
trapping_norm = normalize_series(station_weather_df["weather_trapping_score"])
urban_norm = normalize_series(station_weather_df["urban_pressure_score"])
road_norm = normalize_series(station_weather_df["road_density_proxy"])

station_weather_df["environmental_risk_score"] = (
    0.45 * aqi_norm
    + 0.25 * trapping_norm
    + 0.20 * urban_norm
    + 0.10 * road_norm
) * 100

station_weather_df["environmental_risk_score"] = station_weather_df[
    "environmental_risk_score"
].round(2)

def environmental_risk_category(score):
    if pd.isna(score):
        return "Unknown"
    elif score < 25:
        return "Low"
    elif score < 50:
        return "Moderate"
    elif score < 75:
        return "High"
    else:
        return "Critical"


station_weather_df["environmental_risk_category"] = station_weather_df[
    "environmental_risk_score"
].apply(environmental_risk_category)

print("Environmental risk score calculated.")

display(
    station_weather_df.sort_values(
        "environmental_risk_score",
        ascending=False
    )[
        [
            "state",
            "city",
            "station_name",
            "reported_aqi",
            "reported_aqi_category",
            "weather_trapping_score",
            "urban_pressure_score",
            "road_density_proxy",
            "environmental_risk_score",
            "environmental_risk_category"
        ]
    ].head(20)
)

Environmental risk score calculated.


,state,city,station_name,reported_aqi,reported_aqi_category,weather_trapping_score,urban_pressure_score,road_density_proxy,environmental_risk_score,environmental_risk_category
50,Bihar,Patna,"Muradpur, Patna - BSPCB",500.00,Severe,80.67,60,55,82.98,Critical
180,Karnataka,Bengaluru,"City Railway Station, Bengaluru - KSPCB",500.00,Severe,61.96,65,100,82.62,Critical
84,Delhi,Delhi,"Chandni Chowk, Delhi - IITM",500.00,Severe,54.61,65,100,79.55,Critical
82,Delhi,Delhi,"CRRI Mathura Road, Delhi - IITM",500.00,Severe,54.54,65,100,79.52,Critical
81,Delhi,Delhi,"Burari Crossing, Delhi - IITM",500.00,Severe,56.53,65,85,78.48,Critical
48,Bihar,Patna,"Govt. High School Shikarpur, Patna - BSPCB",437.74,Severe,82.59,60,55,77.99,Critical
126,Gujarat,Ahmedabad,"SAC ISRO Satellite, Ahmedabad - IITM",500.00,Severe,68.01,60,55,77.70,Critical
148,Haryana,Faridabad,"Sector 30, Faridabad - HSPCB",500.00,Severe,53.63,70,65,76.59,Critical
147,Haryana,Faridabad,"Sector 11, Faridabad - HSPCB",500.00,Severe,52.73,70,65,76.21,Critical
462,Uttar Pradesh,Noida,"Sector - 62, Noida - IITM",487.55,Severe,55.43,70,65,76.18,Critical


In [22]:
station_weather_df["weather_data_available"] = station_weather_df[
    numeric_weather_columns
].notna().sum(axis=1) >= 4

station_weather_df["geospatial_data_level"] = "Proxy geospatial features"

station_weather_df["notebook_02_processed_time"] = datetime.now().strftime(
    "%Y-%m-%d %H:%M:%S"
)

print("Weather data availability:")
print(station_weather_df["weather_data_available"].value_counts())

print("\nGeospatial data level:")
print(station_weather_df["geospatial_data_level"].value_counts())

Weather data availability:
weather_data_available
True    500
Name: count, dtype: int64

Geospatial data level:
geospatial_data_level
Proxy geospatial features    500
Name: count, dtype: int64


In [23]:
final_columns = [
    "station_id",
    "station_name",
    "city",
    "state",
    "lat",
    "lon",
    "timestamp",
    "snapshot_fetch_time",

    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "CO",
    "O3",
    "NH3",

    "aqi",
    "aqi_category",
    "dominant_pollutant",
    "aqi_is_valid",
    "aqi_quality_flag",
    "reported_aqi",
    "reported_aqi_category",
    "reported_dominant_pollutant",

    "weather_time",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "wind_speed_10m",
    "wind_direction_10m",
    "surface_pressure",
    "cloud_cover",
    "weather_trapping_score",
    "weather_trapping_category",
    "weather_data_available",

    "geo_region_proxy",
    "landuse_type_proxy",
    "landuse_data_source",
    "urban_pressure_score",
    "urban_pressure_category",
    "road_density_proxy",
    "road_density_category",
    "road_density_data_source",
    "geospatial_data_level",

    "environmental_risk_score",
    "environmental_risk_category",
    "notebook_02_processed_time"
]

existing_final_columns = [
    col for col in final_columns
    if col in station_weather_df.columns
]

station_weather_final = station_weather_df[existing_final_columns].copy()

print("Final dataframe shape:", station_weather_final.shape)
display(station_weather_final.head())

Final dataframe shape: (500, 47)


,station_id,station_name,city,state,lat,lon,timestamp,snapshot_fetch_time,PM2.5,PM10,...,landuse_data_source,urban_pressure_score,urban_pressure_category,road_density_proxy,road_density_category,road_density_data_source,geospatial_data_level,environmental_risk_score,environmental_risk_category,notebook_02_processed_time
0,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-16 05:00:00,2026-07-16 05:38:19,54.0,172.0,...,Proxy from station/city naming,15,Low urban pressure,20,Low road density proxy,Proxy from city/station naming,Proxy geospatial features,51.86,High,2026-07-16 10:51:20
1,andhra_pradesh__anantapur__gulzarpet_anantapur...,"Gulzarpet, Anantapur - APPCB",Anantapur,Andhra Pradesh,14.675886,77.593027,2026-07-16 05:00:00,2026-07-16 05:38:19,85.0,76.0,...,Proxy from station/city naming,15,Low urban pressure,20,Low road density proxy,Proxy from city/station naming,Proxy geospatial features,34.66,Moderate,2026-07-16 10:51:20
2,andhra_pradesh__chittoor__gangineni_cheruvu_ch...,"Gangineni Cheruvu, Chittoor - APPCB",Chittoor,Andhra Pradesh,13.204880,79.097889,2026-07-16 05:00:00,2026-07-16 05:38:19,70.0,64.0,...,Proxy from station/city naming,15,Low urban pressure,20,Low road density proxy,Proxy from city/station naming,Proxy geospatial features,37.09,Moderate,2026-07-16 10:51:20
3,andhra_pradesh__eluru__district_court_eluru_appcb,"District Court, Eluru - APPCB",Eluru,Andhra Pradesh,16.711754,81.092095,2026-07-16 05:00:00,2026-07-16 05:38:19,49.0,49.0,...,Proxy from station/city naming,15,Low urban pressure,20,Low road density proxy,Proxy from city/station naming,Proxy geospatial features,37.99,Moderate,2026-07-16 10:51:20
4,andhra_pradesh__guntur__rajendra_nagar_north_g...,"Rajendra Nagar North, Guntur - APPCB",Guntur,Andhra Pradesh,16.316553,80.413302,2026-07-16 05:00:00,2026-07-16 05:38:19,52.0,52.0,...,Proxy from station/city naming,15,Low urban pressure,20,Low road density proxy,Proxy from city/station naming,Proxy geospatial features,31.32,Moderate,2026-07-16 10:51:20


In [24]:
WEATHER_GEOSPATIAL_OUTPUT_PATH = PROCESSED_DIR / "station_weather_geospatial.csv"

station_weather_final.to_csv(
    WEATHER_GEOSPATIAL_OUTPUT_PATH,
    index=False
)

print("Saved weather + geospatial output:")
print(WEATHER_GEOSPATIAL_OUTPUT_PATH)

print("Rows:", len(station_weather_final))
print("Columns:", len(station_weather_final.columns))

Saved weather + geospatial output:
C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed\station_weather_geospatial.csv
Rows: 500
Columns: 47


In [25]:
city_weather_summary = (
    station_weather_final
    .groupby(["state", "city"], as_index=False)
    .agg(
        station_count=("station_id", "nunique"),
        valid_station_count=("aqi_is_valid", "sum"),
        mean_reported_aqi=("reported_aqi", "mean"),
        max_reported_aqi=("reported_aqi", "max"),
        mean_weather_trapping_score=("weather_trapping_score", "mean"),
        mean_urban_pressure_score=("urban_pressure_score", "mean"),
        mean_road_density_proxy=("road_density_proxy", "mean"),
        mean_environmental_risk_score=("environmental_risk_score", "mean")
    )
)

round_columns = [
    "mean_reported_aqi",
    "max_reported_aqi",
    "mean_weather_trapping_score",
    "mean_urban_pressure_score",
    "mean_road_density_proxy",
    "mean_environmental_risk_score"
]

for col in round_columns:
    city_weather_summary[col] = city_weather_summary[col].round(2)

CITY_WEATHER_SUMMARY_PATH = PROCESSED_DIR / "city_weather_geospatial_summary.csv"

city_weather_summary.to_csv(
    CITY_WEATHER_SUMMARY_PATH,
    index=False
)

print("Saved city-level summary:")
print(CITY_WEATHER_SUMMARY_PATH)

display(
    city_weather_summary.sort_values(
        "mean_environmental_risk_score",
        ascending=False
    ).head(20)
)

Saved city-level summary:


C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed\city_weather_geospatial_summary.csv


,state,city,station_count,valid_station_count,mean_reported_aqi,max_reported_aqi,mean_weather_trapping_score,mean_urban_pressure_score,mean_road_density_proxy,mean_environmental_risk_score
61,Gujarat,Vadodara,1,1,500.00,500.00,63.98,60.00,55.00,76.02
70,Haryana,Faridabad,3,3,445.19,500.00,53.03,70.00,65.00,71.23
37,Bihar,Patna,6,6,369.74,500.00,80.21,60.00,55.00,70.66
237,Uttar Pradesh,Ghaziabad,5,5,456.34,500.00,57.22,60.00,55.00,69.13
60,Gujarat,Surat,2,2,418.87,437.74,65.00,60.00,55.00,68.89
251,Uttar Pradesh,Noida,4,4,397.87,487.55,54.97,70.00,65.00,67.64
155,Maharashtra,Pune,5,5,423.16,500.00,60.28,60.00,55.00,67.32
245,Uttar Pradesh,Lucknow,6,6,380.95,456.42,69.09,60.00,55.00,67.07
54,Gujarat,Ahmedabad,8,8,378.28,500.00,67.28,60.00,55.00,66.06
264,West Bengal,Kolkata,7,7,345.45,450.19,74.28,60.00,55.00,65.93


In [26]:
run_summary = {
    "notebook": "02_weather_geospatial_integration.ipynb",
    "run_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "input_file": str(STATION_INPUT_PATH),
    "input_rows": int(len(station_df)),
    "output_rows": int(len(station_weather_final)),
    "unique_stations": int(station_weather_final["station_id"].nunique()),
    "unique_cities": int(station_weather_final["city"].nunique()),
    "unique_states": int(station_weather_final["state"].nunique()),
    "weather_available_rows": int(station_weather_final["weather_data_available"].sum()),
    "weather_missing_rows": int((~station_weather_final["weather_data_available"]).sum()),
    "output_file": str(WEATHER_GEOSPATIAL_OUTPUT_PATH),
    "city_summary_file": str(CITY_WEATHER_SUMMARY_PATH)
}

RUN_SUMMARY_PATH = REPORTS_DIR / "notebook_02_run_summary.json"

with open(RUN_SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(run_summary, file, indent=4)

print("Run summary saved:")
print(RUN_SUMMARY_PATH)

print(json.dumps(run_summary, indent=4))

Run summary saved:
C:\Users\Lenovo\Desktop\CleanAir_AI\reports\notebook_02_run_summary.json
{
    "notebook": "02_weather_geospatial_integration.ipynb",
    "run_time": "2026-07-16 10:51:20",
    "input_file": "C:\\Users\\Lenovo\\Desktop\\CleanAir_AI\\data\\processed\\station_intelligence_latest.csv",
    "input_rows": 500,
    "output_rows": 500,
    "unique_stations": 500,
    "unique_cities": 264,
    "unique_states": 29,
    "weather_available_rows": 500,
    "weather_missing_rows": 0,
    "output_file": "C:\\Users\\Lenovo\\Desktop\\CleanAir_AI\\data\\processed\\station_weather_geospatial.csv",
    "city_summary_file": "C:\\Users\\Lenovo\\Desktop\\CleanAir_AI\\data\\processed\\city_weather_geospatial_summary.csv"
}


In [27]:
required_output_files = [
    WEATHER_GEOSPATIAL_OUTPUT_PATH,
    CITY_WEATHER_SUMMARY_PATH,
    RUN_SUMMARY_PATH
]

missing_files = [
    path for path in required_output_files
    if not path.exists()
]

if missing_files:
    print("Missing files:")

    for path in missing_files:
        print(path)

    raise FileNotFoundError("Some expected output files were not created.")

print("Notebook 02 completed successfully.")
print("Created files:")

for path in required_output_files:
    print("-", path)

Notebook 02 completed successfully.
Created files:
- C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed\station_weather_geospatial.csv
- C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed\city_weather_geospatial_summary.csv
- C:\Users\Lenovo\Desktop\CleanAir_AI\reports\notebook_02_run_summary.json
